# Symmetry-Centric WBO Matcher: Pseudocode and Full Audit

This notebook is the executable audit for the current matcher. It starts with pseudocode before any implementation claims.

Everything is separated into four labels:

- `USER_ALGORITHM`: directly from the algorithm statement in this notebook.
- `IMPLEMENTATION`: how the current code realizes it.
- `HYPOTHESIS`: an implementation assumption that is not part of the core algorithm.
- `FAILURE`: observed behavior that does not satisfy the expected result.

The notebook regenerates the six standard HTML traces together.

## 1. USER_ALGORITHM Pseudocode

```text
DATA:
    Molecule = weighted WBO graph
        atoms = integer indices
        node label = element
        edge/complete-pair weight = WBO

    FragmentState:
        R_atoms: set of reactant atoms in current fragment
        heap: frontier R edges with WBO >= graph_floor, high WBO first
        candidates: symmetry-centric compressed matches from R_atoms into target atoms
        one_hop_boundary: failed frontier edges used as boundary evidence

    CandidateState:
        Must represent symmetry as correlated atom sets / blocks.
        Must not require full 1-1 bijection enumeration.
        Witness mapping is only a witness, not the represented symmetry state.
        multiplicity records correlated alternatives when context resolves an
        active block to one witness but the alternatives remain symmetry-equivalent.
```

```text
MATCH_EXTENSION(candidates, fragment, new_R_atom n):
    For each compressed candidate state C:
        For each compatible target atom or target symmetry block v:
            Check element compatibility.
            Check target reuse/injectivity in symmetry-aware way.
            Check complete WBO vector:
                for every r in fragment.R_atoms:
                    abs(WBO_R[n, r] - WBO_T[v, C(r)]) <= iso_tol
                where C(r) may be a correlated symmetry assignment.

    Return a compressed CandidateState set.

    Possible outcomes:
        many -> many: grow, keep compressed symmetry
        many -> 1:    grow, symmetry resolved by new context
        many -> 0:    do not choose a bijection; mark one-hop boundary
        1 -> many:    grow; previous atom was a symmetry center
        1 -> 1:       grow
        1 -> 0:       if heap later exhausts, finalize current fragment relation
```

```text
GROW_FRAGMENT(seed_R):
    Initialize fragment with seed_R.
    Initialize candidates with all compatible target seed atoms, grouped by symmetry.
    Initialize heap with all R frontier edges WBO >= graph_floor.

    while heap is not empty:
        edge = pop highest-WBO frontier edge R[u]--R[n]

        if n already in fragment:
            continue

        new_candidates = MATCH_EXTENSION(candidates, fragment, n)

        if new_candidates is non-empty:
            commit n into fragment
            candidates = new_candidates
            push n's frontier edges into heap
        else:
            record one-hop boundary edge R[u]--R[n]
            do not add n

    LOCK_FRAGMENT:
        heap is fully consumed
        finalize fragment relation
        preserve symmetry inside fragment and one-hop boundary evidence
```

```text
MATCH_MOLECULE(R, T):
    Run 3 seed orderings for each match.
    For each seed ordering:
        grow and lock fragments until all possible R atoms are handled.
    Score candidate molecule alignments by least broken + formed bonds.
    Return compressed alignment candidates and best representative.
```

```text
R-P MECHANISM DISCOVERY:
    Run MATCH_MOLECULE(R, P) with no cut.
    For every R edge with WBO >= graph_floor:
        cut that edge only in the R growth graph
        run MATCH_MOLECULE(R_cut, P) with 3 seeds
    Classify bond breaking/forming from the full WBO matrices.
    Keep mechanisms with least broken + formed bonds.
    Dedupe mechanisms by symmetry-aware bond breaking/forming pattern.
```

```text
TS DISCOVERY / RANKING:
    First match R->P to identify mechanisms.
    For each mechanism:
        core atoms = atoms involved in broken/forming bonds.

    Match R->TS and P->TS together, without bond cut sweep.
    Use both to identify TS core atom options and core symmetry.
    Enumerate only core atom mappings.
    Preserve correlation between core atoms; never shuffle correlated atoms independently.

    Score candidate TS core mappings by WBO-weighted vibration alignment on the
    mechanism's forming/breaking bonds. Select imaginary mode by the same metric,
    because this is an initial-guess TS rather than an optimized TS.
```


In [1]:
from __future__ import annotations
import ast, json, math, subprocess, sys, time
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent.resolve()
SRC = ROOT / 'src'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd

from rxn_core.frag import build_graph, classify_bonds, write_xyz_str, expand_mapping
from rxn_core.alignment import match_wbo_graphs, cut_edges_above_floor
from rxn_core.growth import grow_island
from rxn_core.matcher import (
    _SymBlock, _SymCand, _cand_map, _cand_possible_p_atoms,
    _extend_sym_cands, _refine_sym_assignments, _boundary_signature,
    _color_refine_orbits, _edge_wbo, _growth_edge_supported,
)
from build_bgcp_views_v2 import load, WORK
from trace_html import HTML

pd.set_option('display.max_rows', 240)
pd.set_option('display.max_colwidth', 260)

checks = []
def check(name, condition, detail=''):
    checks.append({'check': name, 'pass': bool(condition), 'detail': detail})
    print(('PASS' if condition else 'FAIL'), name, detail)

print('ROOT =', ROOT)
print('HEAD =', subprocess.check_output(['git','rev-parse','--short','HEAD'], cwd=ROOT, text=True).strip())


ROOT = /Users/yunhengz/codex_AAM/rxn_core
HEAD = c52a78d


## 2. IMPLEMENTATION Map Against Pseudocode

In [2]:
IMPLEMENTATION_MAP = [
    ('Molecule WBO graph', 'build_graph + graph.graph["wbo_matrix"] + _edge_wbo', 'IMPLEMENTATION', 'Uses complete WBO matrix for pair checks.'),
    ('Fragment heap', 'grow_island heap + _push_edges_from', 'IMPLEMENTATION', 'Heap contains R edges with WBO >= graph_floor.'),
    ('Compressed symmetry candidates', '_SymCand + _SymBlock', 'IMPLEMENTATION', 'Blocks store correlated R atom set and P atom pool.'),
    ('Extension validity', '_extend_sym_cands + _support_witness_for_value', 'IMPLEMENTATION', 'Checks complete WBO vector to all fragment atoms.'),
    ('One-hop boundary', 'deferred_edges + _boundary_signature', 'IMPLEMENTATION', 'Failed frontier edge is recorded and included in dedupe signature.'),
    ('Fragment lock', 'grow_island returns only after heap exhaustion', 'IMPLEMENTATION', 'Notebook checks no old early-lock string remains.'),
    ('3 seed match', 'match_wbo_graphs(..., n_seeds=3)', 'IMPLEMENTATION', 'Public match function evaluates three seed orderings.'),
    ('R-P cut every edge above floor', 'cut_edges_above_floor', 'IMPLEMENTATION', 'Returns all R pairs with WBO >= graph_floor.'),
    ('WL orbit hierarchy', '_color_refine_orbits', 'HYPOTHESIS', 'Compression heuristic; not itself the user algorithm.'),
    ('Symmetry repair representative', 'symmetry_repair_mapping inside match_wbo_graphs', 'HYPOTHESIS', 'Chooses representative inside compressed symmetry; must not hide growth failures.'),
    ('TS core edge floor', 'ts_core_pool in build_bgcp_views_v2', 'HYPOTHESIS', 'TS-specific policy, not R-P growth algorithm.'),
]
display(pd.DataFrame(IMPLEMENTATION_MAP, columns=['pseudo item','code','label','note']))

,pseudo item,code,label,note
0,Molecule WBO graph,"build_graph + graph.graph[""wbo_matrix""] + _edge_wbo",IMPLEMENTATION,Uses complete WBO matrix for pair checks.
1,Fragment heap,grow_island heap + _push_edges_from,IMPLEMENTATION,Heap contains R edges with WBO >= graph_floor.
2,Compressed symmetry candidates,_SymCand + _SymBlock,IMPLEMENTATION,Blocks store correlated R atom set and P atom pool.
3,Extension validity,_extend_sym_cands + _support_witness_for_value,IMPLEMENTATION,Checks complete WBO vector to all fragment atoms.
4,One-hop boundary,deferred_edges + _boundary_signature,IMPLEMENTATION,Failed frontier edge is recorded and included in dedupe signature.
5,Fragment lock,grow_island returns only after heap exhaustion,IMPLEMENTATION,Notebook checks no old early-lock string remains.
6,3 seed match,"match_wbo_graphs(..., n_seeds=3)",IMPLEMENTATION,Public match function evaluates three seed orderings.
7,R-P cut every edge above floor,cut_edges_above_floor,IMPLEMENTATION,Returns all R pairs with WBO >= graph_floor.
8,WL orbit hierarchy,_color_refine_orbits,HYPOTHESIS,Compression heuristic; not itself the user algorithm.
9,Symmetry repair representative,symmetry_repair_mapping inside match_wbo_graphs,HYPOTHESIS,Chooses representative inside compressed symmetry; must not hide growth failures.


## 3. Concrete Synthetic Fixtures

In [3]:
def wbo_matrix(n, edges):
    M = np.zeros((n, n), float)
    for i, j, w in edges:
        M[i, j] = M[j, i] = float(w)
    return M

def g(elements, edges, floor=0.2):
    return build_graph(elements, wbo_matrix(len(elements), edges), bond_cut=floor)

def represented_count(c):
    sc = c if isinstance(c, _SymCand) else _SymCand(c)
    count = int(getattr(sc, 'multiplicity', 1))
    for b in sc.blocks:
        count *= math.factorial(len(b.p_atoms)) // math.factorial(len(b.p_atoms) - len(b.r_atoms))
    return count

def block_expr(b):
    n, k = len(b.p_atoms), len(b.r_atoms)
    if k == 0 or n <= 1:
        return '1'
    if k == 1:
        return str(n)
    if k == n:
        return f'{n}!'
    return f'P({n},{k})'

def represented_expr(c):
    sc = c if isinstance(c, _SymCand) else _SymCand(c)
    factors = []
    if getattr(sc, 'multiplicity', 1) != 1:
        factors.append(str(sc.multiplicity))
    factors.extend(e for e in (block_expr(b) for b in sc.blocks) if e != '1')
    return ' * '.join(factors) if factors else '1'

def show_candidates(cands):
    rows = []
    for c in cands or []:
        sc = c if isinstance(c, _SymCand) else _SymCand(c)
        rows.append({
            'witness': dict(sorted(sc.mapping.items())),
            'multiplicity': int(getattr(sc, 'multiplicity', 1)),
            'alternate_witnesses': len(getattr(sc, 'alternates', ())),
            'alternate_sample': [dict(items) for items, _ in getattr(sc, 'alternates', ())[:2]],
            'represented': represented_expr(sc),
            'represented_count': represented_count(sc),
            'blocks': [{'r_atoms': b.r_atoms, 'p_atoms': b.p_atoms, 'extendable': b.extendable, 'open': b.open, 'assignments': block_expr(b)} for b in sc.blocks],
            'possible_p': sorted(_cand_possible_p_atoms(sc)),
        })
    return pd.DataFrame(rows)

def run_extend(label, elements_R, edges_R, elements_P, edges_P, fragment, cand, n, iso_tol=0.5, anchor_u=None):
    gR = g(elements_R, edges_R)
    gP = g(elements_P, edges_P)
    out = _extend_sym_cands(
        [cand], set(fragment), n, gR, gP, {}, iso_tol, None,
        p_orbits=_color_refine_orbits(gP), r_orbits=_color_refine_orbits(gR),
        anchor_u=anchor_u,
        anchor_wbo=(_edge_wbo(gR, anchor_u, n) if anchor_u is not None else None),
    )
    print('\n' + label)
    display(show_candidates(out))
    return out, gR, gP

check('growth edge is exact WBO tolerance, accepts 1.384 vs 0.906 at tol 0.5', _growth_edge_supported(1.38410676283019, 0.906139543205486, 0.5))
check('growth edge rejects 1.384 vs 0.0 at tol 0.5', not _growth_edge_supported(1.38410676283019, 0.0, 0.5))

PASS growth edge is exact WBO tolerance, accepts 1.384 vs 0.906 at tol 0.5 
PASS growth edge rejects 1.384 vs 0.0 at tol 0.5 


## 4. Degenerate Hierarchical Symmetry Fixtures

This section is the concrete audit for the failure mode where a symmetry center is grown first and a child atom is attached later.  `Pd(CH3)4` is intentionally small: after choosing `Pd-C`, the represented target choices must be four; after choosing one `C-H`, the parent-carbon choice and the methyl-H choice are correlated and must represent `4 * 3 = 12` assignments without enumerating full bijections.


In [4]:
def tetramethyl_metal_graph():
    elements = ['Pd']
    edges = []
    for i in range(4):
        c = 1 + 4*i
        elements.append('C')
        edges.append((0, c, 0.8))
        for j in range(3):
            h = c + 1 + j
            elements.append('H')
            edges.append((c, h, 1.0))
    return elements, edges

mels, medges = tetramethyl_metal_graph()
mgR = g(mels, medges)
mgP = g(mels, medges)
mr_orbits = _color_refine_orbits(mgR)
mp_orbits = _color_refine_orbits(mgP)
print('Orbit groups:')
orbit_rows = []
for oid in sorted(set(mr_orbits.values())):
    atoms = [a for a, o in sorted(mr_orbits.items()) if o == oid]
    orbit_rows.append({'orbit': oid, 'atoms': atoms, 'elements': [mels[a] for a in atoms]})
display(pd.DataFrame(orbit_rows))

metal_seed = [_SymCand({0: 0})]
mc = _extend_sym_cands(
    metal_seed, {0}, 1, mgR, mgP, {}, 0.5, None,
    p_orbits=mp_orbits, r_orbits=mr_orbits, anchor_u=0, anchor_wbo=0.8)
print('\nPd-C from Pd seed: should represent 4 choices, as one compressed block')
display(show_candidates(mc))
check('Pd(CH3)4 Pd-C represented count is 4', len(mc or []) == 1 and represented_count(mc[0]) == 4)

mcc = _extend_sym_cands(
    mc, {0, 1}, 5, mgR, mgP, {}, 0.5, None,
    p_orbits=mp_orbits, r_orbits=mr_orbits, anchor_u=0, anchor_wbo=0.8)
print('\nTwo Pd-C edges: should represent P(4,2)=12 correlated carbon assignments')
display(show_candidates(mcc))
check('Pd(CH3)4 two Pd-C represented count is 12', len(mcc or []) == 1 and represented_count(mcc[0]) == 12)

mch = _extend_sym_cands(
    mc, {0, 1}, 2, mgR, mgP, {}, 0.5, None,
    p_orbits=mp_orbits, r_orbits=mr_orbits, anchor_u=1, anchor_wbo=1.0)
print('\nPd-C-H: child H must keep parent C degeneracy, representing 4*3=12')
display(show_candidates(mch))
check('Pd(CH3)4 Pd-C-H represented count is 12', len(mch or []) == 1 and represented_count(mch[0]) == 12)

mchh = _extend_sym_cands(
    mch, {0, 1, 2}, 3, mgR, mgP, {}, 0.5, None,
    p_orbits=mp_orbits, r_orbits=mr_orbits, anchor_u=1, anchor_wbo=1.0)
print('\nPd-C-H-H: second H on same methyl gives 4*P(3,2)=24')
display(show_candidates(mchh))
check('Pd(CH3)4 Pd-C-H-H represented count is 24', len(mchh or []) == 1 and represented_count(mchh[0]) == 24)


Orbit groups:


,orbit,atoms,elements
0,0,"[1, 5, 9, 13]","[C, C, C, C]"
1,1,"[2, 3, 4, 6, 7, 8, 10, 11, 12, 14, 15, 16]","[H, H, H, H, H, H, H, H, H, H, H, H]"
2,2,[0],[Pd]



Pd-C from Pd seed: should represent 4 choices, as one compressed block


,witness,multiplicity,alternate_witnesses,alternate_sample,represented,represented_count,blocks,possible_p
0,"{0: 0, 1: 1}",1,0,[],4,4,"[{'r_atoms': (1,), 'p_atoms': (1, 5, 9, 13), 'extendable': True, 'open': True, 'assignments': '4'}]","[0, 1, 5, 9, 13]"


PASS Pd(CH3)4 Pd-C represented count is 4 

Two Pd-C edges: should represent P(4,2)=12 correlated carbon assignments


...

PASS Pd(CH3)4 two Pd-C represented count is 12 

Pd-C-H: child H must keep parent C degeneracy, representing 4*3=12


,witness,multiplicity,alternate_witnesses,alternate_sample,represented,represented_count,blocks,possible_p
0,"{0: 0, 1: 5, 2: 6}",12,11,"[{0: 0, 1: 5, 2: 7}, {0: 0, 1: 5, 2: 8}]",12,12,[],"[0, 5, 6]"


PASS Pd(CH3)4 Pd-C-H represented count is 12 

Pd-C-H-H: second H on same methyl gives 4*P(3,2)=24


,witness,multiplicity,alternate_witnesses,alternate_sample,represented,represented_count,blocks,possible_p
0,"{0: 0, 1: 5, 2: 6, 3: 7}",12,0,[],12 * 2,24,"[{'r_atoms': (3,), 'p_atoms': (7, 8), 'extendable': True, 'open': True, 'assignments': '2'}]","[0, 5, 6, 7, 8]"


PASS Pd(CH3)4 Pd-C-H-H represented count is 24 


## 5. Hidden Alternate Witness and `cut_all_cands`

`cut_all_cands` means the popped frontier atom has no target that satisfies the complete WBO vector for any represented candidate variant.  This section tests the failure mode from the trace: a compressed candidate may have hidden alternate witnesses.  Those alternates must be available to later frontier growth; otherwise a scalar multiplicity would be display-only and wrong.


In [5]:
# A minimal hidden-alternate case.
# Fragment = R0,R1.  The primary witness maps R0->P0,R1->P1 and cannot add R2->P2,
# because R0-R2 is 1.0 but P0-P2 is 0.0.  The alternate witness R0->P1,R1->P0 can add it.
hr = g(['C','C','O'], [(0,2,1.0)])
hp = g(['C','C','O'], [(1,2,1.0)])
hidden = _SymCand({0:0, 1:1}, multiplicity=2, alternates=[({0:1, 1:0}, 1)])
print('before extension: one compressed candidate, two represented witnesses')
display(show_candidates([hidden]))
hout = _extend_sym_cands([hidden], {0,1}, 2, hr, hp, {}, 0.5, None, anchor_u=0, anchor_wbo=1.0)
print('after extension: the alternate witness is used')
display(show_candidates(hout))
check('hidden alternate witness can extend later frontier atom', len(hout or []) == 1 and _cand_map(hout[0]) == {0:1, 1:0, 2:2})

# Complete-vector cut_all_cands case: anchor edge can match, but another already-fragmented atom forbids it.
cr = g(['C','C','O'], [(0,2,1.0)])
cp = g(['C','C','O'], [(0,2,1.0),(1,2,1.1)])
cout = _extend_sym_cands([_SymCand({0:0, 1:1})], {0,1}, 2, cr, cp, {}, 0.5, None, anchor_u=0, anchor_wbo=1.0)
print('complete-vector rejection: anchor R0-R2 matches P0-P2, but non-anchor R1-R2=0 vs P1-P2=1.1')
display(show_candidates(cout))
check('complete WBO vector can reject a popped edge through a non-anchor atom', len(cout or []) == 0)


before extension: one compressed candidate, two represented witnesses


,witness,multiplicity,alternate_witnesses,alternate_sample,represented,represented_count,blocks,possible_p
0,"{0: 0, 1: 1}",2,1,"[{0: 1, 1: 0}]",2,2,[],"[0, 1]"


after extension: the alternate witness is used


,witness,multiplicity,alternate_witnesses,alternate_sample,represented,represented_count,blocks,possible_p
0,"{0: 1, 1: 0, 2: 2}",1,0,[],1,1,[],"[0, 1, 2]"


PASS hidden alternate witness can extend later frontier atom 
complete-vector rejection: anchor R0-R2 matches P0-P2, but non-anchor R1-R2=0 vs P1-P2=1.1


""


PASS complete WBO vector can reject a popped edge through a non-anchor atom 


## 6. All Growth Scenario Examples

In [6]:
# 1 -> 0
out_10, _, _ = run_extend(
    '1 -> 0: no target H has WBO within tolerance',
    ['C','H'], [(0,1,1.0)],
    ['C','H','H'], [],
    fragment={0}, cand=_SymCand({0:0}), n=1, iso_tol=0.5, anchor_u=0)
check('1->0 returns zero extension candidates', len(out_10 or []) == 0)

# 1 -> 1
out_11, _, _ = run_extend(
    '1 -> 1: exactly one H target valid',
    ['C','H'], [(0,1,1.0)],
    ['C','H','H'], [(0,1,1.0)],
    fragment={0}, cand=_SymCand({0:0}), n=1, iso_tol=0.5, anchor_u=0)
check('1->1 maps R1 to P1', bool(out_11) and _cand_map(out_11[0]).get(1) == 1)

# 1 -> many
out_1m, _, _ = run_extend(
    '1 -> many: two equivalent target H atoms are one block',
    ['C','H'], [(0,1,1.0)],
    ['C','H','H'], [(0,1,1.0),(0,2,1.0)],
    fragment={0}, cand=_SymCand({0:0}), n=1, iso_tol=0.5, anchor_u=0)
check('1->many is compressed as one candidate', len(out_1m or []) == 1)
check('1->many candidate contains P pool {1,2}', bool(out_1m) and {1,2}.issubset(_cand_possible_p_atoms(out_1m[0])))

# many -> 1: existing block R1 in P1/P2; adding O forces R1->P2 and R2->P3.
gR = g(['C','H','O'], [(0,1,1.0),(1,2,1.0)])
gP = g(['C','H','H','O'], [(0,1,1.0),(0,2,1.0),(2,3,1.0)])
base = _SymCand({0:0}).with_new_block(1, (1,2), extendable=True)
out_m1 = _extend_sym_cands([base], {0,1}, 2, gR, gP, {}, 0.5, None, p_orbits=_color_refine_orbits(gP), r_orbits=_color_refine_orbits(gR), anchor_u=1, anchor_wbo=1.0)
print('\nmany -> 1: new atom resolves prior symmetry')
display(show_candidates(out_m1))
check('many->1 resolves to R1->P2 and R2->P3', bool(out_m1) and _cand_map(out_m1[0]).get(1) == 2 and _cand_map(out_m1[0]).get(2) == 3)

# many -> many: two H atoms around same C remain a correlated complete block.
gR = g(['C','H','H'], [(0,1,1.0),(0,2,1.0)])
gP = g(['C','H','H'], [(0,1,1.0),(0,2,1.0)])
base = _SymCand({0:0}).with_new_block(1, (1,2), extendable=True)
out_mm = _extend_sym_cands([base], {0,1}, 2, gR, gP, {}, 0.5, None, p_orbits=_color_refine_orbits(gP), r_orbits=_color_refine_orbits(gR), anchor_u=0, anchor_wbo=1.0)
print('\nmany -> many: correlated H block remains represented')
display(show_candidates(out_mm))
check('many->many keeps one compressed/correlated candidate', len(out_mm or []) == 1)

# many -> 0: same prior H symmetry block, but adding O has no compatible O edge.
m0_gR = g(['C','H','O'], [(0,1,1.0),(1,2,1.0)])
m0_badP = g(['C','H','H','O'], [(0,1,1.0),(0,2,1.0)])
m0_base = _SymCand({0:0}).with_new_block(1, (1,2), extendable=True)
out_m0 = _extend_sym_cands([m0_base], {0,1}, 2, m0_gR, m0_badP, {}, 0.5, None, p_orbits=_color_refine_orbits(m0_badP), r_orbits=_color_refine_orbits(m0_gR), anchor_u=1, anchor_wbo=1.0)
print('\nmany -> 0: no valid target for next atom')
display(show_candidates(out_m0))
check('many->0 returns zero extension candidates', len(out_m0 or []) == 0)


1 -> 0: no target H has WBO within tolerance


""


PASS 1->0 returns zero extension candidates 

1 -> 1: exactly one H target valid


,witness,multiplicity,alternate_witnesses,alternate_sample,represented,represented_count,blocks,possible_p
0,"{0: 0, 1: 1}",1,0,[],1,1,[],"[0, 1]"


PASS 1->1 maps R1 to P1 

1 -> many: two equivalent target H atoms are one block


...

PASS many->1 resolves to R1->P2 and R2->P3 

many -> many: correlated H block remains represented


,witness,multiplicity,alternate_witnesses,alternate_sample,represented,represented_count,blocks,possible_p
0,"{0: 0, 1: 2, 2: 1}",1,0,[],2!,2,"[{'r_atoms': (1, 2), 'p_atoms': (1, 2), 'extendable': True, 'open': False, 'assignments': '2!'}]","[0, 1, 2]"


PASS many->many keeps one compressed/correlated candidate 

many -> 0: no valid target for next atom


""


PASS many->0 returns zero extension candidates 


## 7. One-Hop Boundary and Lock Examples

In [7]:
# 1->0 with heap exhaustion should finalize current fragment and record boundary.
gR = g(['C','H'], [(0,1,1.0)])
gP = g(['C','H'], [])
events = []
isos = grow_island(gR, gP, seed=0, mapping={}, iso_tol=0.5, events=events, p_orbits=_color_refine_orbits(gP), r_orbits=_color_refine_orbits(gR))
print('events')
for i, e in enumerate(events):
    print(i, e.get('type'), e.get('edge'), e.get('reason'), e.get('fragment'), e.get('heap_remaining'))
print('isos', isos, 'deferred', [getattr(x, 'deferred_edges', None) for x in isos])
check('1->0 at heap exhaustion locks seed fragment', len(isos) == 1 and dict(isos[0]) == {0:0})
check('1->0 records one-hop boundary edge', len(isos) == 1 and (0,1) in isos[0].deferred_edges)

# Boundary must alter dedupe signature.
c = _SymCand({0:0})
sig0 = _boundary_signature(c, gR, gP, fragment={0}, deferred_edges=(), r_orbits=_color_refine_orbits(gR), p_orbits=_color_refine_orbits(gP), locked_mapping={0:0})
sig1 = _boundary_signature(c, gR, gP, fragment={0}, deferred_edges={(0,1)}, r_orbits=_color_refine_orbits(gR), p_orbits=_color_refine_orbits(gP), locked_mapping={0:0})
print('boundary empty', sig0)
print('boundary one-hop', sig1)
check('one-hop boundary changes signature', sig0 != sig1)
impl_files = list((ROOT/'src/rxn_core/alignment').glob('*.py')) + list((ROOT/'src/rxn_core/growth').glob('*.py')) + list((ROOT/'src/rxn_core/matcher').glob('*.py'))
impl_text = '\n'.join(path.read_text() for path in impl_files)
check('no early-lock implementation string remains', 'set_unique_len1_during_BFS' not in impl_text)

events
0 seed_start None None [0] None
1 pop {'frag_atom': 0, 'ext_atom': 1, 'wbo': 1.0, 'ext_element': 'H'} None None None
2 consumed {'frag_atom': 0, 'ext_atom': 1, 'wbo': 1.0, 'ext_element': 'H'} cut_all_cands [0] 0
3 seed_end None None [0] None
isos [{0: 0}] deferred [frozenset({(0, 1)})]
PASS 1->0 at heap exhaustion locks seed fragment 
PASS 1->0 records one-hop boundary edge 
boundary empty ()
boundary one-hop ((1, 'H', ((0, 0, 5),), ((0, 0, 5),), ((('H', 1, ((0, 0),), ()), 1),)),)
PASS one-hop boundary changes signature 
PASS no early-lock implementation string remains 


## 8. R-P Cut Set: Every Edge Above Floor

In [8]:
w = wbo_matrix(4, [(0,1,0.19),(0,2,0.2),(0,3,0.7),(1,2,1.0)])
cut_set = cut_edges_above_floor(w, floor=0.2)
print('cut_set', cut_set)
check('cut set includes every WBO >= floor', set(cut_set) == {(0,2),(0,3),(1,2)})
check('cut set excludes below floor', (0,1) not in cut_set)

cut_set ((0, 2), (0, 3), (1, 2))
PASS cut set includes every WBO >= floor 
PASS cut set excludes below floor 


## 9. Six Standard Test Cases Generated Together Using MATCH_MOLECULE

In [9]:
CASES = [
    ('pr14_alkene', 'pr14.Pd_hydroamination_JOC2025_TS3_step2_alkene_inserion'),
    ('pr7_ts910', 'pr7.V.dodh_ts910'),
    ('pr17_ts6a', 'pr17.carbene.ins_ts6a'),
]
TRACE_OUT = ROOT / 'notebooks' / 'alignment_traces'
TRACE_OUT.mkdir(parents=True, exist_ok=True)

def trace_events_for_best(result):
    if result.best is None:
        return []
    if result.best.events:
        return result.best.events
    for cand in result.candidates:
        if cand.seed_index == result.best.seed_index and cand.branch_index == 0 and cand.events:
            return cand.events
    return []

def mapping_summary_html(result, elR, elT, right_short):
    rows = []
    for rank, c in enumerate(result.candidates[:120], 1):
        mapping = ', '.join(f'R[{r}]({elR[r]})&rarr;{right_short}[{t}]({elT[t]})' for r, t in sorted(c.mapping.items()))
        rows.append(
            f'<tr><td>{rank}</td><td>{c.seed_index}</td><td>{c.branch_index}</td>'
            f'<td>{len(c.broken)}/{len(c.formed)}</td><td>{len(c.mapping)}</td>'
            f'<td>{len(c.deferred_edges)}</td>'
            f'<td style="font-family:ui-monospace,monospace;font-size:11px;word-break:break-all">{mapping}</td></tr>'
        )
    return '<table class="mt" style="border-collapse:collapse;width:100%"><tr><th>rank</th><th>seed</th><th>branch</th><th>br/fm</th><th>mapped</th><th>deferred</th><th>mapping</th></tr>' + ''.join(rows) + '</table>'

def write_match_trace(step_label, step_name, target_label, target_dir, right_short):
    sd = WORK / step_name
    elR, xyzR, wboR = load(sd/'R')
    elT, xyzT, wboT = load(sd/target_dir)
    t0 = time.time()
    result = match_wbo_graphs(elR, wboR, elT, wboT, xyzR=xyzR, xyzP=xyzT, n_seeds=3, capture_events=True)
    elapsed = time.time() - t0
    best = result.best
    events = trace_events_for_best(result)
    path = TRACE_OUT / f'{step_label}_{target_label.replace("-","_")}_alignment_trace.html'
    title = f'{step_label} {target_label} match trace best={len(best.broken)}/{len(best.formed)} seed={best.seed_index} branch={best.branch_index} candidates={len(result.candidates)} elapsed={elapsed:.3f}s'
    html = HTML.format(
        title=title,
        left_title='Reactant',
        right_title='Product' if target_label == 'R-P' else 'Ground truth TS',
        left_short='R', right_short=right_short,
        xyzR_json=json.dumps(write_xyz_str(elR, xyzR, comment='R')),
        xyzP_json=json.dumps(write_xyz_str(elT, xyzT, comment=right_short)),
        events_json=json.dumps(events),
        wboR_json=json.dumps(wboR.tolist()),
        wboP_json=json.dumps(wboT.tolist()),
        elements_R_json=json.dumps(elR),
        elements_P_json=json.dumps(elT),
        mappings_summary_html=mapping_summary_html(result, elR, elT, right_short),
    )
    path.write_text(html)
    counts = Counter((len(c.broken), len(c.formed)) for c in result.candidates)
    count_rows = [
        {'broken': br, 'formed': fm, 'count': count}
        for (br, fm), count in sorted(counts.items())
    ]
    return {
        'step': step_label, 'target': target_label, 'path': str(path),
        'best_broken': len(best.broken), 'best_formed': len(best.formed),
        'best_seed': best.seed_index, 'best_branch': best.branch_index,
        'n_candidates': len(result.candidates), 'elapsed_s': round(elapsed, 3),
        'counts': count_rows,
        'events': len(events),
    }

six = []
for step_label, step_name in CASES:
    six.append(write_match_trace(step_label, step_name, 'R-P', 'P', 'P'))
    six.append(write_match_trace(step_label, step_name, 'R-GT', 'sp_groundtruth', 'GT'))
(TRACE_OUT / 'latest_alignment_trace_summary.json').write_text(json.dumps(six, indent=2))
summary = pd.DataFrame([{k:v for k,v in row.items() if k != 'counts'} for row in six])
display(summary)
for row in six:
    print(row['step'], row['target'], row['counts'])
check('six traces generated', len(six) == 6)
check('pr7 R-P best is 2/2', any(r['step']=='pr7_ts910' and r['target']=='R-P' and (r['best_broken'], r['best_formed']) == (2,2) for r in six))

,step,target,path,best_broken,best_formed,best_seed,best_branch,n_candidates,elapsed_s,events
0,pr14_alkene,R-P,/Users/yunhengz/codex_AAM/rxn_core/notebooks/alignment_traces/pr14_alkene_R_P_alignment_trace.html,2,3,0,0,6,15.425,243
1,pr14_alkene,R-GT,/Users/yunhengz/codex_AAM/rxn_core/notebooks/alignment_traces/pr14_alkene_R_GT_alignment_trace.html,0,0,0,0,3,7.188,227
2,pr7_ts910,R-P,/Users/yunhengz/codex_AAM/rxn_core/notebooks/alignment_traces/pr7_ts910_R_P_alignment_trace.html,2,2,0,0,10,0.186,80
3,pr7_ts910,R-GT,/Users/yunhengz/codex_AAM/rxn_core/notebooks/alignment_traces/pr7_ts910_R_GT_alignment_trace.html,2,2,1,1,16,0.228,85
4,pr17_ts6a,R-P,/Users/yunhengz/codex_AAM/rxn_core/notebooks/alignment_traces/pr17_ts6a_R_P_alignment_trace.html,1,1,0,0,3,5.960,310
5,pr17_ts6a,R-GT,/Users/yunhengz/codex_AAM/rxn_core/notebooks/alignment_traces/pr17_ts6a_R_GT_alignment_trace.html,0,0,0,0,3,6.237,292


pr14_alkene R-P [{'broken': 2, 'formed': 3, 'count': 6}]
pr14_alkene R-GT [{'broken': 0, 'formed': 0, 'count': 3}]
pr7_ts910 R-P [{'broken': 2, 'formed': 2, 'count': 6}, {'broken': 3, 'formed': 3, 'count': 4}]
pr7_ts910 R-GT [{'broken': 2, 'formed': 2, 'count': 2}, {'broken': 3, 'formed': 3, 'count': 2}, {'broken': 9, 'formed': 9, 'count': 6}, {'broken': 10, 'formed': 10, 'count': 6}]
pr17_ts6a R-P [{'broken': 1, 'formed': 1, 'count': 3}]
pr17_ts6a R-GT [{'broken': 0, 'formed': 0, 'count': 3}]
PASS six traces generated 
PASS pr7 R-P best is 2/2 


## 10. Explicit Uncertainty / Risk Ledger

In [10]:
risk_rows = [
    {'label':'HYPOTHESIS', 'item':'WL/Morgan color refinement', 'detail':'Used as hierarchy/compression. It is not a proof of automorphism equivalence.'},
    {'label':'HYPOTHESIS', 'item':'symmetry_repair_mapping', 'detail':'Chooses a representative inside compressed symmetry for bond-change scoring. It must not hide growth failure.'},
    {'label':'HYPOTHESIS', 'item':'support search cap', 'detail':'SYM_SUPPORT_MAX_STATES bounds witness search. A cap hit is implementation failure/uncertainty, not algorithmic invalidity.'},
    {'label':'HYPOTHESIS', 'item':'TS_CORE_EDGE_FLOOR', 'detail':'TS matching policy, not R-P mechanism discovery.'},
    {'label':'IMPLEMENTATION', 'item':'3 seed match', 'detail':'Implemented in match_wbo_graphs and used by align_from_arrays.'},
    {'label':'IMPLEMENTATION', 'item':'cut every edge above floor', 'detail':'cut_edges_above_floor and BGCP_CUT_FLOOR=0.2.'},
]
display(pd.DataFrame(risk_rows))

,label,item,detail
0,HYPOTHESIS,WL/Morgan color refinement,Used as hierarchy/compression. It is not a proof of automorphism equivalence.
1,HYPOTHESIS,symmetry_repair_mapping,Chooses a representative inside compressed symmetry for bond-change scoring. It must not hide growth failure.
2,HYPOTHESIS,support search cap,"SYM_SUPPORT_MAX_STATES bounds witness search. A cap hit is implementation failure/uncertainty, not algorithmic invalidity."
3,HYPOTHESIS,TS_CORE_EDGE_FLOOR,"TS matching policy, not R-P mechanism discovery."
4,IMPLEMENTATION,3 seed match,Implemented in match_wbo_graphs and used by align_from_arrays.
5,IMPLEMENTATION,cut every edge above floor,cut_edges_above_floor and BGCP_CUT_FLOOR=0.2.


## 11. Final Check Summary

In [11]:
checks_df = pd.DataFrame(checks)
display(checks_df)
print('PASS', int(checks_df['pass'].sum()), '/', len(checks_df))
if not checks_df['pass'].all():
    display(checks_df[~checks_df['pass']])

,check,pass,detail
0,"growth edge is exact WBO tolerance, accepts 1.384 vs 0.906 at tol 0.5",True,
1,growth edge rejects 1.384 vs 0.0 at tol 0.5,True,
2,Pd(CH3)4 Pd-C represented count is 4,True,
3,Pd(CH3)4 two Pd-C represented count is 12,True,
4,Pd(CH3)4 Pd-C-H represented count is 12,True,
5,Pd(CH3)4 Pd-C-H-H represented count is 24,True,
6,hidden alternate witness can extend later frontier atom,True,
7,complete WBO vector can reject a popped edge through a non-anchor atom,True,
8,1->0 returns zero extension candidates,True,
9,1->1 maps R1 to P1,True,


PASS 23 / 23
